In [1]:

# Import necessary libraries
import numpy as np
import pickle
import pandas as pd
from scipy.optimize import minimize_scalar
from numba import njit
import matplotlib.pyplot as plt
from collections import defaultdict
import time

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully")
print(f"NumPy version: {np.__version__}")


Libraries imported successfully
NumPy version: 1.26.4


In [2]:

# Load the pre-computed artifacts
print("Loading pre-computed artifacts...")

# Load f_canon_rand prime coefficients for N=10^7
with open('f_canon_rand_primes_N1e7.pkl', 'rb') as f:
 f_canon_data = pickle.load(f)
 primes_1e7 = f_canon_data['primes']
 coefficients_1e7 = f_canon_data['coefficients']
 prime_to_coeff_1e7 = f_canon_data['prime_to_coefficient']
 
print(f"Loaded {len(primes_1e7)} primes up to {primes_1e7[-1]}")
print(f"Sample coefficients: {coefficients_1e7[:5]}")

# Load omega values for N=10^7
with open('omega_values_N1e7.pkl', 'rb') as f:
 omega_1e7 = pickle.load(f)
print(f"Loaded omega values for N=10^7, shape: {omega_1e7.shape}")

# Load omega values for N=10^6
with open('omega_values_N1e6.pkl', 'rb') as f:
 omega_1e6 = pickle.load(f)
print(f"Loaded omega values for N=10^6, shape: {omega_1e6.shape}")

print("\nAll artifacts loaded successfully")


Loading pre-computed artifacts...


KeyError: 'coefficients'

In [3]:

# Check the structure of the loaded data
print("Keys in f_canon_data:", f_canon_data.keys())


Keys in f_canon_data: dict_keys(['primes', 'a_p', 'prime_coeff_dict', 'seed', 'max_n', 'generation_method', 'formula'])


In [4]:

# Load the pre-computed artifacts with correct keys
print("Loading pre-computed artifacts...")

# Load f_canon_rand prime coefficients for N=10^7
with open('f_canon_rand_primes_N1e7.pkl', 'rb') as f:
 f_canon_data = pickle.load(f)
 primes_1e7 = f_canon_data['primes']
 coefficients_1e7 = f_canon_data['a_p'] # Correct key
 prime_to_coeff_1e7 = f_canon_data['prime_coeff_dict'] # Correct key
 
print(f"Loaded {len(primes_1e7)} primes up to {primes_1e7[-1]}")
print(f"Sample coefficients: {coefficients_1e7[:5]}")
print(f"Verification - all |a_p| = 1: {np.allclose(np.abs(coefficients_1e7), 1.0)}")

# Load omega values for N=10^7
with open('omega_values_N1e7.pkl', 'rb') as f:
 omega_1e7 = pickle.load(f)
print(f"\nLoaded omega values for N=10^7, shape: {omega_1e7.shape}")

# Load omega values for N=10^6
with open('omega_values_N1e6.pkl', 'rb') as f:
 omega_1e6 = pickle.load(f)
print(f"Loaded omega values for N=10^6, shape: {omega_1e6.shape}")

print("\nAll artifacts loaded successfully")


Loading pre-computed artifacts...


Loaded 664579 primes up to 9999991
Sample coefficients: [-0.70506063+0.70914702j 0.95243384-0.30474544j -0.11289421-0.99360701j
 -0.81394263-0.58094525j 0.55677833+0.83066112j]
Verification - all |a_p| = 1: True

Loaded omega values for N=10^7, shape: (10000000,)
Loaded omega values for N=10^6, shape: (1000000,)

All artifacts loaded successfully


In [5]:

# Generate the full f_canon_rand coefficients for all n up to N
# For a canonical random multiplicative function: a_n = product of a_p for all primes p dividing n

def generate_f_canon_rand_coefficients(N, primes, a_p_values, omega_values):
 """
 Generate coefficients for f_canon_rand using the canonical multiplicative structure.
 a_n = product of a_p for primes p dividing n (with multiplicity)
 
 Parameters:
 -----------
 N : int
 Maximum n value
 primes : array
 Array of primes
 a_p_values : array
 Coefficients a_p for each prime
 omega_values : array
 Omega(n) values (0-indexed: omega_values[i] = Omega(i+1))
 
 Returns:
 --------
 a_n : array of complex128
 Coefficients for n=1 to N
 """
 print(f"Generating f_canon_rand coefficients up to N={N}...")
 start_time = time.time()
 
 # Create prime-to-coefficient mapping
 prime_to_coeff = {p: a_p_values[i] for i, p in enumerate(primes)}
 
 # Initialize coefficients array
 a_n = np.ones(N, dtype=np.complex128)
 
 # For each n, compute the product of a_p for all prime factors
 for n in range(2, N + 1):
 n_temp = n
 for p in primes:
 if p > n_temp:
 break
 while n_temp % p == 0:
 a_n[n-1] *= prime_to_coeff[p]
 n_temp //= p
 if n_temp == 1:
 break
 
 elapsed = time.time() - start_time
 print(f"Coefficient generation completed in {elapsed:.2f} seconds")
 
 return a_n

# This is computationally intensive, so let's first test with N=10^6
print("Testing coefficient generation for N=10^6...")
a_n_1e6 = generate_f_canon_rand_coefficients(
 N=10**6,
 primes=primes_1e7,
 a_p_values=coefficients_1e7,
 omega_values=omega_1e6
)

print(f"\nGenerated coefficients for N=10^6")
print(f"a_1 = {a_n_1e6[0]} (should be 1)")
print(f"Sample coefficients: {a_n_1e6[:10]}")
print(f"All |a_n| = 1: {np.allclose(np.abs(a_n_1e6), 1.0)}")


Testing coefficient generation for N=10^6...
Generating f_canon_rand coefficients up to N=1000000...


Coefficient generation completed in 749.39 seconds

Generated coefficients for N=10^6
a_1 = (1+0j) (should be 1)
Sample coefficients: [ 1. +0.j -0.70506063+0.70914702j 0.95243384-0.30474544j
 -0.005779 -0.9999833j -0.11289421-0.99360701j -0.45541428+0.89027964j
 -0.81394263-0.58094525j 0.71320973+0.7009507j 0.81426043-0.58049974j
 0.78421072+0.6204946j ]
All |a_n| = 1: True


In [6]:

# The coefficient generation is too slow for N=10^7 (would take ~2 hours)
# We need to use a more efficient approach with Numba

@njit
def generate_f_canon_rand_coefficients_numba(N, primes, a_p_real, a_p_imag):
 """
 Numba-accelerated generation of f_canon_rand coefficients.
 
 Parameters:
 -----------
 N : int
 Maximum n value
 primes : array
 Array of primes
 a_p_real, a_p_imag : arrays
 Real and imaginary parts of a_p coefficients
 
 Returns:
 --------
 a_n_real, a_n_imag : arrays
 Real and imaginary parts of a_n coefficients
 """
 a_n_real = np.ones(N, dtype=np.float64)
 a_n_imag = np.zeros(N, dtype=np.float64)
 
 for n in range(2, N + 1):
 n_temp = n
 coeff_real = 1.0
 coeff_imag = 0.0
 
 for i in range(len(primes)):
 p = primes[i]
 if p > n_temp:
 break
 
 while n_temp % p == 0:
 # Complex multiplication: (coeff_real + i*coeff_imag) * (a_p_real[i] + i*a_p_imag[i])
 new_real = coeff_real * a_p_real[i] - coeff_imag * a_p_imag[i]
 new_imag = coeff_real * a_p_imag[i] + coeff_imag * a_p_real[i]
 coeff_real = new_real
 coeff_imag = new_imag
 n_temp //= p
 
 if n_temp == 1:
 break
 
 a_n_real[n-1] = coeff_real
 a_n_imag[n-1] = coeff_imag
 
 return a_n_real, a_n_imag

# Test with N=10^5 first
print("Testing Numba coefficient generation for N=10^5...")
start = time.time()
a_n_real_test, a_n_imag_test = generate_f_canon_rand_coefficients_numba(
 10**5,
 primes_1e7,
 np.real(coefficients_1e7),
 np.imag(coefficients_1e7)
)
elapsed = time.time() - start
print(f"Numba generation for N=10^5 completed in {elapsed:.2f} seconds")

# Verify correctness
a_n_test = a_n_real_test + 1j * a_n_imag_test
print(f"a_1 = {a_n_test[0]} (should be 1)")
print(f"All |a_n| = 1: {np.allclose(np.abs(a_n_test), 1.0)}")

# Compare with previous result for first 10 values
print(f"\nComparison with previous method (first 10 values):")
print(f"Match: {np.allclose(a_n_test[:10], a_n_1e6[:10])}")


Testing Numba coefficient generation for N=10^5...


Numba generation for N=10^5 completed in 0.98 seconds
a_1 = (1+0j) (should be 1)
All |a_n| = 1: True

Comparison with previous method (first 10 values):
Match: True


In [7]:

# Excellent! The Numba version is ~750x faster. Now generate for N=10^7
print("Generating f_canon_rand coefficients for N=10^7...")
print("This may take a few minutes...")
start = time.time()

a_n_real_1e7, a_n_imag_1e7 = generate_f_canon_rand_coefficients_numba(
 10**7,
 primes_1e7,
 np.real(coefficients_1e7),
 np.imag(coefficients_1e7)
)

elapsed = time.time() - start
print(f"Numba generation for N=10^7 completed in {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

# Combine into complex array
a_n_1e7 = a_n_real_1e7 + 1j * a_n_imag_1e7

print(f"\nGenerated coefficients for N=10^7")
print(f"a_1 = {a_n_1e7[0]} (should be 1)")
print(f"Sample coefficients: {a_n_1e7[:10]}")
print(f"All |a_n| = 1: {np.allclose(np.abs(a_n_1e7), 1.0)}")


Generating f_canon_rand coefficients for N=10^7...
This may take a few minutes...
Numba generation for N=10^7 completed in 751.19 seconds (12.52 minutes)

Generated coefficients for N=10^7
a_1 = (1+0j) (should be 1)
Sample coefficients: [ 1. +0.j -0.70506063+0.70914702j 0.95243384-0.30474544j
 -0.005779 -0.9999833j -0.11289421-0.99360701j -0.45541428+0.89027964j
 -0.81394263-0.58094525j 0.71320973+0.7009507j 0.81426043-0.58049974j
 0.78421072+0.6204946j ]
All |a_n| = 1: True


In [8]:

# Now implement the Numba-accelerated Dirichlet polynomial evaluation
@njit
def eval_dirichlet_numba(t, a_n_real, a_n_imag):
 """
 Numba-accelerated evaluation of Dirichlet polynomial.
 D_F(t; N) = Σ_{n=1}^N a_n / n^{1/2 + it}
 
 Parameters:
 -----------
 t : float
 Imaginary part of s = 1/2 + it
 a_n_real, a_n_imag : arrays
 Real and imaginary parts of coefficients
 
 Returns:
 --------
 D_real, D_imag : float
 Real and imaginary parts of D_F(t; N)
 """
 N = len(a_n_real)
 D_real = 0.0
 D_imag = 0.0
 
 for n in range(1, N + 1):
 # n^{-1/2 - it} = n^{-1/2} * e^{-it*log(n)}
 # = n^{-1/2} * (cos(-t*log(n)) + i*sin(-t*log(n)))
 # = n^{-1/2} * (cos(t*log(n)) - i*sin(t*log(n)))
 
 inv_sqrt_n = 1.0 / np.sqrt(n)
 log_n = np.log(n)
 angle = t * log_n
 
 cos_angle = np.cos(angle)
 sin_angle = np.sin(angle)
 
 # Complex multiplication: a_n * n^{-1/2-it}
 # = (a_n_real + i*a_n_imag) * inv_sqrt_n * (cos_angle - i*sin_angle)
 term_real = a_n_real[n-1] * inv_sqrt_n * cos_angle + a_n_imag[n-1] * inv_sqrt_n * sin_angle
 term_imag = a_n_imag[n-1] * inv_sqrt_n * cos_angle - a_n_real[n-1] * inv_sqrt_n * sin_angle
 
 D_real += term_real
 D_imag += term_imag
 
 return D_real, D_imag

# Test the function
print("Testing Numba Dirichlet evaluation...")
test_t = 1e7
start = time.time()
D_real, D_imag = eval_dirichlet_numba(test_t, a_n_real_1e7, a_n_imag_1e7)
elapsed = time.time() - start

D_test = D_real + 1j * D_imag
print(f"D_F({test_t}; 10^7) = {D_test}")
print(f"|D_F| = {np.abs(D_test)}")
print(f"Evaluation time: {elapsed:.2f} seconds")


Testing Numba Dirichlet evaluation...


D_F(10000000.0; 10^7) = (-0.4702382897259417+1.0254140390023663j)
|D_F| = 1.1280948543928053
Evaluation time: 0.90 seconds


In [9]:

# The evaluation is still too slow. Let's check the timing more carefully
# and see if we can optimize further

print("Timing test for single evaluation at N=10^7...")
test_times = []
for _ in range(5):
 start = time.time()
 D_real, D_imag = eval_dirichlet_numba(1e7, a_n_real_1e7, a_n_imag_1e7)
 elapsed = time.time() - start
 test_times.append(elapsed)
 print(f" Evaluation time: {elapsed:.3f} seconds")

avg_time = np.mean(test_times)
print(f"\nAverage evaluation time: {avg_time:.3f} seconds")
print(f"Estimated time for 1000 points: {avg_time * 1000 / 60:.1f} minutes")
print(f"Estimated time for 10000 points: {avg_time * 10000 / 60:.1f} minutes")

# This is too slow. We need to reduce the number of evaluations
print("\nThe grid search at N=10^7 would take too long.")
print("We need a more efficient strategy.")


TimeoutError: Code execution timed out after 798 seconds